In [1]:
from ipywidgets import Output
output = Output()
output

Output()

## Simulation

In [2]:
from nanover.openmm import OpenMMSimulation
from nanover.mdanalysis.converter import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

## iMD runner + client

In [3]:
from nanover.app import OmniRunner
from nanover.websocket import NanoverImdClient

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="MOVEABLE RESTRAINTS")
imd_runner.load(0)
imd = imd_runner.app_server.imd

client = NanoverImdClient.from_runner(imd_runner)

In [4]:
with client.root_selection.modify() as selection:
    selection.renderer = "cartoon"

In [5]:
def notify_all(message):
    for command in imd_runner.app_server.commands:
        if "/notify" in command:
            imd_runner.app_server.run_command(command, dict(message=message))

## Utilties

In [6]:
from nanover.jupyter import NanoverJupyterUtilities, Mode

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.use_recording_commands()
utilities.use_interaction_modes()

## Restraints

In [7]:
from nanover.app import RenderingSelection
from nanover.imd.imd_state import INTERACTION_PREFIX, ParticleInteraction

RESTRAINT_PREFIX = f"{INTERACTION_PREFIX}MOVEABLE-RESTRAINT."

NEXT_RESTRAINT_INDEX = 0
MOVEABLE_RESTRAINTS: dict[str, ParticleInteraction] = {}
SELECTIONS: dict[str, RenderingSelection] = {}
ACTIVE_RESTRAINTS: set[str] = set()

RENDERER = {
    "render": {
        "particle.scale": .1,
        "bond.scale": 0.0,
        "type": "ball and stick"
    },
    "color": "CornflowerBlue",
}

def add_restraint(indexes):
    global NEXT_RESTRAINT_INDEX
    indexes = [int(index) for index in indexes]

    key = f"{RESTRAINT_PREFIX}{NEXT_RESTRAINT_INDEX}"
    NEXT_RESTRAINT_INDEX += 1
    restraint = ParticleInteraction((0, 0, 0), indexes)
    restraint.scale = 1000
    restraint.interaction_type = "spring"
    MOVEABLE_RESTRAINTS[key] = restraint

    with client.create_selection(key, indexes).modify() as selection:
        selection.renderer = RENDERER
        selection.velocity_reset = True
        selection.interaction_method = "group"
        SELECTIONS[key] = selection

    enable_restraint(key)

    return key


def remove_restraint(key):
    disable_restraint(key)
    client.remove_selection(SELECTIONS[key])
    MOVEABLE_RESTRAINTS.pop(key, None)


def refresh_restraints():
    with client.root_selection.modify() as selection:
        selection.renderer = "cartoon"

    for key in ACTIVE_RESTRAINTS:
        restraint = MOVEABLE_RESTRAINTS[key]
        indexes = [int(index) for index in restraint.particles]

        with client.create_selection(key, indexes).modify() as selection:
            selection.renderer = RENDERER
            selection.velocity_reset = True
            selection.interaction_method = "group"
            SELECTIONS[key] = selection

        imd.insert_interaction(key, restraint)


def enable_restraint(key):
    if key in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.add(key)
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    restraint = MOVEABLE_RESTRAINTS[key]
    restraint.position = positions[restraint.particles].sum(axis=0) / len(restraint.particles)
    imd.insert_interaction(key, restraint)


def disable_restraint(key):
    if key not in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.remove(key)
    imd.remove_interaction(key)

## Commands

In [8]:
def clear_restraints():
    for restraint in list(MOVEABLE_RESTRAINTS.keys()):
        remove_restraint(restraint)
    MOVEABLE_RESTRAINTS.clear()

imd_runner.app_server.register_command("user/restraints/clear", clear_restraints)

In [9]:
class InteractMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if key.startswith(RESTRAINT_PREFIX):
            return

        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                disable_restraint(key)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        if key.startswith(RESTRAINT_PREFIX):
            return

        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                enable_restraint(key)

utilities.add_interaction_mode(InteractMode, "move")

In [10]:
class ToggleMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if key.startswith(RESTRAINT_PREFIX):
            return

        interacted_indexes = set(interaction.particles)
        removed = False
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                remove_restraint(key)
                removed = True
        if not removed:
            add_restraint(interaction.particles)

utilities.add_interaction_mode(ToggleMode, "toggle")

In [11]:
import numpy as np
import time
from follower import Follower, Group
from nanover.imd import ParticleInteraction

REPLAY_SPEED = .01
CHECKPOINT = None
FOLLOWER: Follower | None = None
GROUPS: list[Group] = []


def clone_omni_omm(omni_omm):
    """
    Create an independent copy of a given NanoVer OpenMMSimulation.
    """
    from nanover.openmm import serializer

    # get the underlying openmm simulation object
    omm = omni_omm.simulation

    # clone it by serializing and then deserializing
    data = serializer.serialize_simulation(omm)
    omm_clone = serializer.deserialize_simulation(data)

    # wrap the cloned openmm simulation in OpenMMSimulation
    omni_omm_clone = OpenMMSimulation.from_simulation(omm_clone)

    # give it different name
    omni_omm_clone.name = omni_omm.name + "'"

    return omni_omm_clone


def set_checkpoint():
    global CHECKPOINT, GROUPS
    CHECKPOINT = clone_omni_omm(imd_runner.simulation)
    GROUPS = []
    InteractMode.enter()
    notify_all("CHECKPOINT SET")


def reset_to_checkpoint():
    global FOLLOWER
    if FOLLOWER is not None:
        FOLLOWER.stop()
        FOLLOWER = None
    imd_runner.simulations[0] = CHECKPOINT
    imd_runner.load(0)
    time.sleep(.1)
    refresh_restraints()
    InteractMode.enter()


def replay_from_checkpoint():
    global FOLLOWER

    InteractMode.enter()
    if CHECKPOINT is None:
        notify_all("NO CHECKPOINT")
    else:
        reset_to_checkpoint()
        notify_all(f"REPLAYING FROM CHECKPOINT (SPEED {REPLAY_SPEED})")
        FOLLOWER = Follower.from_runner(imd_runner)
        FOLLOWER.start(GROUPS, speed=REPLAY_SPEED)


def on_interaction_updated(*, key: str, interaction: ParticleInteraction):
    if CHECKPOINT is not None and FOLLOWER is None:
        for group in GROUPS:
            if set(group.particles) == set(interaction.particles):
                positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
                centroid = np.average(positions[interaction.particles], axis=0)
                group.path.add_point(centroid)
                break
        else:
            GROUPS.append(Group(particles=interaction.particles))

imd_runner.app_server.imd.interaction_updated.add_callback(on_interaction_updated)

# add the command to the server,
# commands with names beginning with "user/" are displayed in the VR client
imd_runner.app_server.register_command("user/checkpoint/set", set_checkpoint)
imd_runner.app_server.register_command("user/checkpoint/reset", reset_to_checkpoint)
imd_runner.app_server.register_command("user/checkpoint/replay", replay_from_checkpoint)

def do_slower():
    global REPLAY_SPEED
    REPLAY_SPEED *= .5
    notify_all(f"SPEED {REPLAY_SPEED}")

def do_faster():
    global REPLAY_SPEED
    REPLAY_SPEED *= 2
    notify_all(f"SPEED {REPLAY_SPEED}")

imd_runner.app_server.register_command("user/checkpoint/slower", do_slower)
imd_runner.app_server.register_command("user/checkpoint/faster", do_faster)